# v24 学習ノート ― origin トグル付き LightGBM(train.parquet → OOF/モデル)

- `create_train_parquet_v24.py` が出した train.parquet を読み、**由来(origin)で特徴をトグル**して LightGBM を GroupKFold 学習。
- origin: base(無接頭)/ pf_ / heur_ / beam_ / gsf_。`USE_ORIGIN` で ON/OFF。
- OOF overall RMSE(全行プール=Kaggle指標)を raw と apply_pp で表示 → CV。fold pkl 保存(再開可)。


In [9]:
# ===== 設定 + parquet 読み込み + origin トグルで特徴選択 =====
import json, joblib, warnings, gc
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings("ignore")

ART     = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v24\artifacts_v24_3")
PARQUET = ART / "data" / "train.parquet"
MODELS  = ART / "models"; MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 0.5     # 学習に使う行の割合(1.0=全行)。OOM/時短で下げる。overall評価は1.0推奨。
pp_params = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}   # apply_pp(submitと共有)

# ===== ★ origin トグル(由来ごとに特徴をON/OFF)=====
USE_ORIGIN = {"base": True, "pf": True, "heur": False, "beam": True, "gsf": True}
PREFIXES = {"pf": "pf_", "heur": "heur_", "beam": "beam_", "gsf": "gsf_"}
META = {"well", "id", "target"}

def origin_of(c):
    for o, pre in PREFIXES.items():
        if c.startswith(pre): return o
    return "base"

df = pd.read_parquet(PARQUET)
all_feats = [c for c in df.columns if c not in META]
features = [c for c in all_feats if USE_ORIGIN.get(origin_of(c), True)]
print("origin別 全特徴数:", dict(Counter(origin_of(c) for c in all_feats)))
print("USE_ORIGIN:", USE_ORIGIN, "→ 採用", len(features), "列")
ART.joinpath("features.json").write_text(json.dumps(features))
ART.joinpath("pp_params.json").write_text(json.dumps(pp_params))
ART.joinpath("use_origin.json").write_text(json.dumps(USE_ORIGIN))

if ROW_FRAC < 1.0:
    step = max(1, int(round(1.0 / ROW_FRAC))); df = df.iloc[::step].reset_index(drop=True)
y  = df["target"].to_numpy(np.float32)
g  = df["well"].to_numpy()
lk = df["last_known_tvt"].to_numpy(np.float64) if "last_known_tvt" in df.columns else np.zeros(len(df))
mds = df["md_since"].to_numpy(np.float64) if "md_since" in df.columns else np.zeros(len(df))
pf_anc_col = next((c for c in df.columns if c.startswith("pf_ancc")), None)
pf_res = (df[pf_anc_col].to_numpy(np.float64) - lk) if pf_anc_col else np.zeros(len(df))
X = df[features].astype("float32"); del df; gc.collect()
print("rows", len(X), "| features", len(features), "| wells", pd.unique(g).size, "| pf_anc_col", pf_anc_col)


origin別 全特徴数: {'base': 373, 'beam': 43, 'pf': 391, 'heur': 22, 'gsf': 10}
USE_ORIGIN: {'base': True, 'pf': True, 'heur': False, 'beam': True, 'gsf': True} → 採用 817 列
rows 1891995 | features 817 | wells 773 | pf_anc_col pf_ancc


In [11]:
# ===== 変種: LightGBM×3 + CatBoost×3 + XGBoost×3(v9/v20系, GPU) =====
# ※ GPUが無い/未対応なら: lgb device='cpu'/device_type='cpu', cb task_type='CPU', xgb device='cpu' に。
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f"lightgbm-{i+1}", "lgb", p) for i, p in enumerate(lgb_params)] +
            [(f"catboost-{i+1}", "cb", p) for i, p in enumerate(cb_params)] +
            [(f"xgboost-{i+1}", "xgb", p) for i, p in enumerate(xgb_params)])
print("variants:", [v[0] for v in VARIANTS])


variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [12]:
# ===== GroupKFold(5) で学習 → OOF収集 + fold pkl保存(再開可) =====
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == "lgb":
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse",
              callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == "cb":
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        fp = d / f"fold{k}.pkl"
        if fp.exists():
            m = joblib.load(fp); tag = "load"
        else:
            m = fit_one(kind, params, X.iloc[tr], y[tr], X.iloc[va], y[va]); joblib.dump(m, fp); tag = "fit"
        oof[name][va] = m.predict(X.iloc[va])
        print(f"{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}", flush=True)
        del m; gc.collect()
    print(f"== {name} OOF rmse={root_mean_squared_error(y, oof[name]):.4f}")


Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1686]	valid_0's rmse: 7.19331	valid_0's l2: 51.7437
lightgbm-1 fold0 [fit]: rmse=7.1933
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1825]	valid_0's rmse: 6.84143	valid_0's l2: 46.8052
lightgbm-1 fold1 [fit]: rmse=6.8414
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[3060]	valid_0's rmse: 7.63753	valid_0's l2: 58.3318
lightgbm-1 fold2 [fit]: rmse=7.6375
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[81]	valid_0's rmse: 8.25804	valid_0's l2: 68.1952
lightgbm-1 fold3 [fit]: rmse=8.2580
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[710]	valid_0's rmse: 8.67941	valid_0's l2: 75.3322
lightgbm-1 fold4 [fit]: rmse=8.6794
== lightgbm-1 OOF rmse=7.7514
Training until validation scores don't impr

KeyboardInterrupt: 

In [ ]:
# 7.48

In [8]:
# ===== アンサンブル OOF + apply_pp で overall RMSE(=Kaggle指標, CV) =====
names = [v[0] for v in VARIANTS]
oof_ens = np.mean([oof[n] for n in names], axis=0)

def apply_pp_res(res):
    d = res * (1 - pp_params["w_pf"]) + pf_res * pp_params["w_pf"]   # GBM残差 と PF残差 を blend
    if pp_params.get("tau"):
        d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(pp_params["tau"])))
    return d * float(pp_params.get("alpha", 1.0))

raw = root_mean_squared_error(y, oof_ens)
ppr = root_mean_squared_error(y, apply_pp_res(oof_ens))
print(f"\n=== OOF overall RMSE(全行プール=Kaggle指標) ===")
print(f"  raw(GBMアンサンブルのみ) : {raw:.4f}")
print(f"  + apply_pp              : {ppr:.4f}")
print(f"  USE_ORIGIN = {USE_ORIGIN}")
print("  ※ sg_smooth は per-well なので OOF では未適用(submit で適用)。CV↔LB は ±0.2 のノイズ床込みで見る。")



=== OOF overall RMSE(全行プール=Kaggle指標) ===
  raw(GBMアンサンブルのみ) : 7.4112
  + apply_pp              : 7.4022
  USE_ORIGIN = {'base': True, 'pf': True, 'heur': True, 'beam': True, 'gsf': False}
  ※ sg_smooth は per-well なので OOF では未適用(submit で適用)。CV↔LB は ±0.2 のノイズ床込みで見る。
